# Lab 2  Messy Data & Leakage-Safe Features

**DSA 8401 Applied Machine Learning · Chapter 2**

This notebook works the four Lab 2 tasks on the synthetic mobile-money dump `mobile_money_statements.csv` using the patterns from Chapter 2

1. **Audit**  profile missingness / duplicates / validity violations; classify each
   incomplete column MCAR / MAR / MNAR 
2. **Clean & aggregate**  parse amounts and dates
     - resolve entities to a canonical `customer_id` 
     -  build customer-level RFM, ratio and cyclical features 
3. **Hunt the leak**  correlation screen (Listing 2.11) + a lineage argument to find the two planted leaks.
4. **Pipeline**  one `ColumnTransformer` `Pipeline`, group-aware CV, ROC–AUC with and without the leaks 



In [1]:
import re, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")



In [2]:
# The scoring timestamp: every feature is computed over data strictly BEFORE this
SCORING_TS = pd.Timestamp("2026-08-01 00:00:00")



In [3]:
from pathlib import Path

DATA_PATH = Path("../Data/mobile_money_statements.csv")

raw = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH.resolve())
print("raw shape:", raw.shape)

raw.head(3).T

Dataset path: C:\Users\USER\Documents\MSc_DSA\Module_4\Applied_ML\AML_Book-Data\Data\mobile_money_statements.csv
raw shape: (183600, 18)


,0,1,2
txn_id,TXN0102856,TXN0155490,TXN0019434
msisdn,254700004022,254700013612,254700005862
reg_id,NID101089,NID103658,nid101584
account_name,Joseph Mwangi,Rehema Ndlovu,Mary Mutua
region,Kampala,Mombasa,Arusha
segment,retail,retail,retail
txn_time,2026-06-30 14:24:19,"Jan 29, 2026 01:17 PM","Sep 16, 2025 12:55 AM"
txn_type,send,send,receive
amount,"1,329/- Dr",289/- Dr,"1,571/-"
gps_lat,NaN,1.7747,-3.74336


In [4]:
raw.head(10)

,txn_id,msisdn,reg_id,account_name,region,segment,txn_time,txn_type,amount,gps_lat,gps_lon,agent_id,device_id,counterparty,balance_after,manual_review_score,settlement_status,is_fraud
0,TXN0102856,254700004022,NID101089,Joseph Mwangi,Kampala,retail,2026-06-30 14:24:19,send,"1,329/- Dr",NaN,NaN,AG1014,DV978723,CP6324,8123.0,0.061,settled,0
1,TXN0155490,254700013612,NID103658,Rehema Ndlovu,Mombasa,retail,"Jan 29, 2026 01:17 PM",send,289/- Dr,1.77470,34.70915,AG1084,DV616667,CP2831,6322.0,0.000,settled,0
2,TXN0019434,254700005862,nid101584,Mary Mutua,Arusha,retail,"Sep 16, 2025 12:55 AM",receive,"1,571/-",-3.74336,33.67018,AG1091,DV637734,CP167,15237.0,0.912,reversed,1
3,TXN0049426,254700009223,NID102493,Joseph Okoth,Dar es Salaam,retail,07/04/2026 11:21,send,599/- Dr,NaN,NaN,AG1047,DV666470,CP3382,3322.0,0.095,settled,0
4,TXN0013816,254700014760,nid103963,B. Ssentongo,Mwanza,retail,20/12/2025 11:05,cashout,"(1,307/-)",0.08284,29.02835,NaN,NaN,CP8428,2168.0,0.085,settled,0
5,TXN0068766,254700005080,NID101375,Grace Mutua,Arusha,retail,2026-06-21 12:00:08,send,"2,128/- Dr",-0.28486,33.04071,NaN,DV322595,CP6317,3202.0,0.066,settled,0
6,TXN0006514,254700010321,NID102782,SSENTONGO GRACE,Mwanza,retail,24/07/2026 04:01,bill,"-2,136/-",-3.07853,33.77321,NaN,DV492573,CP7821,4046.0,0.008,settled,0
7,TXN0115001,254700007185,NID101945,Amina Ali,Mombasa,retail,"May 25, 2026 12:17 PM",cashin,"1,026/-",NaN,NaN,AG1026,DV104452,CP6532,3226.0,0.172,settled,0
8,TXN0067557,254700011799,nid103172,G. Kamau,Mombasa,merchant,22/10/2025 22:44,cashin,"1,099/-",2.62258,33.90121,AG1099,DV674871,CP5021,6475.0,0.069,settled,0
9,TXN0086664,254700005680,NID101533,Aisha Ali,Nairobi,retail,2025-09-02 07:48:24,receive,951/-,NaN,NaN,AG1338,DV107852,CP609,4576.0,0.090,pending,0


In [5]:
# eye balling

raw.shape

(183600, 18)

In [6]:
raw.columns

Index(['txn_id', 'msisdn', 'reg_id', 'account_name', 'region', 'segment',
       'txn_time', 'txn_type', 'amount', 'gps_lat', 'gps_lon', 'agent_id',
       'device_id', 'counterparty', 'balance_after', 'manual_review_score',
       'settlement_status', 'is_fraud'],
      dtype='object')

In [7]:
raw.dtypes

txn_id                  object
msisdn                   int64
reg_id                  object
account_name            object
region                  object
segment                 object
txn_time                object
txn_type                object
amount                  object
gps_lat                float64
gps_lon                float64
agent_id                object
device_id               object
counterparty            object
balance_after          float64
manual_review_score    float64
settlement_status       object
is_fraud                 int64
dtype: object

In [8]:
raw.select_dtypes(include="object")

,txn_id,reg_id,account_name,region,segment,txn_time,txn_type,amount,agent_id,device_id,counterparty,settlement_status
0,TXN0102856,NID101089,Joseph Mwangi,Kampala,retail,2026-06-30 14:24:19,send,"1,329/- Dr",AG1014,DV978723,CP6324,settled
1,TXN0155490,NID103658,Rehema Ndlovu,Mombasa,retail,"Jan 29, 2026 01:17 PM",send,289/- Dr,AG1084,DV616667,CP2831,settled
2,TXN0019434,nid101584,Mary Mutua,Arusha,retail,"Sep 16, 2025 12:55 AM",receive,"1,571/-",AG1091,DV637734,CP167,reversed
3,TXN0049426,NID102493,Joseph Okoth,Dar es Salaam,retail,07/04/2026 11:21,send,599/- Dr,AG1047,DV666470,CP3382,settled
4,TXN0013816,nid103963,B. Ssentongo,Mwanza,retail,20/12/2025 11:05,cashout,"(1,307/-)",NaN,NaN,CP8428,settled
...,...,...,...,...,...,...,...,...,...,...,...,...
183595,TXN0066455,NID102893,"Mutua, Grace",Dar es Salaam,retail,27/11/2025 18:15,airtime,(52/-),AG1191,DV964052,CP208,settled
183596,TXN0053459,nid101960,OSEI GRACE,Arusha,retail,"Jul 18, 2026 07:03 AM",send,"(1,003/-)",NaN,DV963852,CP6035,settled
183597,TXN0010742,NID100470,OTIENO FATUMA,Nakuru,retail,05/02/2026 05:33,receive,920/-,AG1081,DV731612,CP8535,settled
183598,TXN0049689,NID100206,Z. Ssentongo,Mombasa,retail,24/03/2026 01:18,bill,"(2,848/-)",AG1163,DV746921,CP8162,reversed


In [9]:
raw['region'].unique()

array(['Kampala', 'Mombasa', 'Arusha', 'Dar es Salaam', 'Mwanza',
       'Nairobi', 'Eldoret', 'Kigali', 'Nakuru', 'Jinja'], dtype=object)

In [10]:
raw.isna().mean().sort_values(ascending=False)

gps_lon                0.362108
gps_lat                0.362108
agent_id               0.136939
device_id              0.059129
msisdn                 0.000000
txn_id                 0.000000
segment                0.000000
reg_id                 0.000000
region                 0.000000
account_name           0.000000
amount                 0.000000
txn_type               0.000000
txn_time               0.000000
counterparty           0.000000
balance_after          0.000000
manual_review_score    0.000000
settlement_status      0.000000
is_fraud               0.000000
dtype: float64

In [11]:
df = raw.copy()

## 1. Audit(Missing  Nos)

Three cheap diagnostics :

- **quantify** the missing fraction per column, 
- **test** whether missingness is informative,
- **inspect co-missingness**. 

We also check duplicates and the validity rules the dump violates.

In [12]:
raw.isna().sum()/raw.shape[0]

txn_id                 0.000000
msisdn                 0.000000
reg_id                 0.000000
account_name           0.000000
region                 0.000000
segment                0.000000
txn_time               0.000000
txn_type               0.000000
amount                 0.000000
gps_lat                0.362108
gps_lon                0.362108
agent_id               0.136939
device_id              0.059129
counterparty           0.000000
balance_after          0.000000
manual_review_score    0.000000
settlement_status      0.000000
is_fraud               0.000000
dtype: float64

In [13]:
# 1a. Missingness fraction per column (Listing 2.1, step 1) ---


miss = raw.isna().mean().sort_values(ascending=False)

print("Missing fraction per column:")

print(miss[miss > 0].round(4).to_string())

Missing fraction per column:
gps_lon      0.3621
gps_lat      0.3621
agent_id     0.1369
device_id    0.0591


In [14]:
# 1b. Duplicates ---
# Exact row duplicates (client retries / log merges) -> drop_duplicates dispatches them.


print("exact duplicate rows :", int(raw.duplicated().sum()))

print("duplicate txn_id      :", int(raw['txn_id'].duplicated().sum()))


# txn_id repeats exactly on the duplicate rows they are true row copies, not new events.

exact duplicate rows : 3600
duplicate txn_id      : 3600


In [15]:
# 1b-extended. Duplicate transaction details ---

print("\n" + "=" * 60)
print("DUPLICATE TRANSACTION DISTRIBUTION")
print("=" * 60)

print("\nDuplicate transaction IDs (txn_id value counts):")
print(
    raw.loc[raw["txn_id"].duplicated(keep=False), "txn_id"]
       .value_counts()
       .head(10)
)


DUPLICATE TRANSACTION DISTRIBUTION

Duplicate transaction IDs (txn_id value counts):
txn_id
TXN0109206    2
TXN0156469    2
TXN0103640    2
TXN0134272    2
TXN0077778    2
TXN0019894    2
TXN0049518    2
TXN0151783    2
TXN0111824    2
TXN0072429    2
Name: count, dtype: int64


In [16]:
# 1c. Validity-rule violations ---

# amount is stored as a STRING with thousands separators, a "/-" suffix, and three
# different debit encodings; it will not cast to a number as-is.

print("amount dtype:", raw['amount'].dtype)

print("amount samples:", raw['amount'].dropna().sample(6, random_state=1).tolist())

# txn_time mixes three formats -> a single to_datetime(format=...) cannot parse all rows.
print("txn_time samples:", raw['txn_time'].sample(6, random_state=2).tolist())

amount dtype: object
amount samples: ['71/- Dr', '4,742/- Dr', '1,701/-', '(227/-)', '(165/-)', '(51/-)']
txn_time samples: ['29/06/2026 04:49', '2025-09-09 05:09:38', 'Mar 20, 2026 02:09 PM', 'May 04, 2025 08:43 PM', '2026-03-30 12:00:04', '03/08/2025 21:49']


In [17]:
# 1c-extended. Detailed validity checks ---

# Numerical summary
print("=" * 60)
print("NUMERICAL FIELD OVERVIEW")
print("=" * 60)
print(raw.describe(include="all").T)

# Balance_after distribution (if this column exists)
print("\nBalance_after detailed percentiles:")
print(raw["balance_after"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

# GPS bounds validation
print("\n" + "=" * 60)
print("GPS COORDINATE VALIDITY")
print("=" * 60)

invalid_lat = ((raw["gps_lat"] < -90) | (raw["gps_lat"] > 90)).sum()
invalid_lon = ((raw["gps_lon"] < -180) | (raw["gps_lon"] > 180)).sum()

print("Invalid latitude :", invalid_lat)
print("Invalid longitude:", invalid_lon)

# Amount head sample
print("\n" + "=" * 60)
print("AMOUNT FIELD SAMPLES (messiness evidence)")
print("=" * 60)
print(raw["amount"].head(20).tolist())

NUMERICAL FIELD OVERVIEW
                        count  unique                    top    freq  \
txn_id                 183600  180000             TXN0066604       2   
msisdn               183600.0     NaN                    NaN     NaN   
reg_id                 183600   11722              NID101450     152   
account_name           183600    1232                A. Osei     441   
region                 183600      10                Mombasa   20108   
segment                183600       3                 retail  147208   
txn_time               183600  175449  Sep 26, 2025 05:40 PM       4   
txn_type               183600       7                   send   44042   
amount                 183600   24833                52/- Dr      61   
gps_lat              117117.0     NaN                    NaN     NaN   
gps_lon              117117.0     NaN                    NaN     NaN   
agent_id               158458     400                 AG1321     471   
device_id              172744  154381  

In [18]:
raw["gps_lat"].notna()

0         False
1          True
2          True
3         False
4          True
          ...  
183595    False
183596    False
183597     True
183598     True
183599     True
Name: gps_lat, Length: 183600, dtype: bool

In [19]:
raw.loc[raw["gps_lat"].notna(), "is_fraud"].mean()

np.float64(0.08387339156569926)

In [20]:
# -1d. Is missingness informative? (the MNAR screen)

# Compare the target rate across the missing indicator for each incomplete column.

for col in ["gps_lat", "agent_id", "device_id"]:

    # Fraud rate when the value is present
    present_rate = raw.loc[raw[col].notna(), "is_fraud"].mean()

    # Fraud rate when the value is missing
    missing_rate = raw.loc[raw[col].isna(), "is_fraud"].mean()

    print(f"\n{col}")
    print(f"  Present : {present_rate:.4f}")
    print(f"  Missing : {missing_rate:.4f}")


gps_lat
  Present : 0.0839
  Missing : 0.0843

agent_id
  Present : 0.0841
  Missing : 0.0839

device_id
  Present : 0.0840
  Missing : 0.0838


In [21]:

# 1e. Can OBSERVED columns explain the missingness? (MAR test) ---

# agent_id: missing fraction by region.


# Percentage of missing agent_id values in each region
print("Missing agent_id by region")
print(
    raw.groupby("region")["agent_id"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

# Percentage of missing gps_lat values in each region
print("\nMissing gps_lat by region")
print(
    raw.groupby("region")["gps_lat"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

# Percentage of missing device_id values in each region
print("\nMissing device_id by region")
print(
    raw.groupby("region")["device_id"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

Missing agent_id by region
region
Arusha           0.452
Dar es Salaam    0.000
Eldoret          0.000
Jinja            0.451
Kampala          0.000
Kigali           0.000
Mombasa          0.000
Mwanza           0.448
Nairobi          0.000
Nakuru           0.000
Name: agent_id, dtype: float64

Missing gps_lat by region
region
Arusha           0.360
Dar es Salaam    0.362
Eldoret          0.357
Jinja            0.365
Kampala          0.365
Kigali           0.367
Mombasa          0.354
Mwanza           0.364
Nairobi          0.360
Nakuru           0.366
Name: gps_lat, dtype: float64

Missing device_id by region
region
Arusha           0.060
Dar es Salaam    0.059
Eldoret          0.058
Jinja            0.056
Kampala          0.061
Kigali           0.061
Mombasa          0.060
Mwanza           0.058
Nairobi          0.059
Nakuru           0.058
Name: device_id, dtype: float64


In [22]:
# 1f. Consistency checks: can we canonicalise messy identifiers? ---

print("\n" + "=" * 60)
print("IDENTIFIER CONSISTENCY & ENTITY RESOLUTION EVIDENCE")
print("=" * 60)

# reg_id consistency
print("\nRegistration ID canonicalisation:")
print(f"Raw unique reg_id: {raw['reg_id'].nunique()}")
print(f"After strip + upper: {raw['reg_id'].astype(str).str.strip().str.upper().nunique()}")

# Region variants
print("\nRegion values (check for duplicates/spelling variants):")
print(sorted(raw["region"].dropna().unique()))

# Multi-SIM evidence: same customer (by reg_id) can own multiple SIMs
print("\nMulti-SIM evidence per customer:")
sims_per_customer = raw.groupby(raw["reg_id"].astype(str).str.strip().str.upper())["msisdn"].nunique()
print(f"Customers with 1 SIM: {(sims_per_customer == 1).sum()}")
print(f"Customers with 2+ SIMs: {(sims_per_customer >= 2).sum()}")
print(f"Max SIMs per customer: {sims_per_customer.max()}")


IDENTIFIER CONSISTENCY & ENTITY RESOLUTION EVIDENCE

Registration ID canonicalisation:
Raw unique reg_id: 11722
After strip + upper: 3991

Region values (check for duplicates/spelling variants):
['Arusha', 'Dar es Salaam', 'Eldoret', 'Jinja', 'Kampala', 'Kigali', 'Mombasa', 'Mwanza', 'Nairobi', 'Nakuru']

Multi-SIM evidence per customer:
Customers with 1 SIM: 3311
Customers with 2+ SIMs: 680
Max SIMs per customer: 3


### 10 Most Significant Data-Quality Issues

The following ten columns were prioritised based on the severity, prevalence, potential impact on modelling, and implications for point-in-time reliability. The classifications and remediation decisions are based on the diagnostic evidence above.

| Column | Main issue | Evidence | Missingness classification | Remediation |
|---|---|---|---|---|
| `gps_lat` | Missingness / timeliness | 36.21% missing | MNAR-like / domain-based | Impute within the modelling pipeline and retain a missingness indicator; avoid treating missing GPS as a valid location. |
| `gps_lon` | Missingness / timeliness | 36.21% missing | MNAR-like / domain-based | Impute within the modelling pipeline and retain a missingness indicator. |
| `agent_id` | Missingness | 13.69% missing; ~45% missing in three regions | MAR | Retain the field, encode missingness explicitly, and impute within the Pipeline. |
| `device_id` | Missingness | 5.91% missing; approximately uniform across regions | Approximately MCAR | Impute using a dedicated missing category/indicator within the Pipeline. |
| `amount` | Validity / representation | Values include `"4,742/- Dr"` and `"(227/-)"` | N/A | Parse sign and numeric magnitude into validated numeric fields; flag unparseable values. |
| `txn_time` | Validity / consistency | Three timestamp formats observed | N/A | Parse all supported formats into a single datetime representation and validate failures. |
| `txn_id` | Uniqueness | 3,600 duplicate transaction IDs/rows | N/A | Remove exact duplicate records before feature construction and retain one canonical transaction record. |
| `reg_id` | Consistency / entity resolution | 11,722 raw IDs → 3,991 canonical IDs | N/A | Strip/standardise identifiers and resolve records to a canonical `customer_id`; use the resolved customer for grouping. |
| `manual_review_score` | Timeliness / leakage | \|correlation\| = 0.981 with target | N/A | Exclude from modelling because it is generated through a downstream review process; enforce write-time eligibility. |
| `settlement_status` | Timeliness / leakage | `reversed` = 100% fraud; `held` = 65.1% | N/A | Exclude from modelling because settlement information is downstream of the scoring event; enforce point-in-time feature eligibility. |

### Data Quality Findings

- MAR -  The missing value depends on another variable that you already have. , yes we can explain
- MNAR-  Missing because of the value itself ,No we cant
- MCAR  - Completely random, No we cant The missing values happened purely by chance.


**Duplicates.** Exactly **3,600** rows are exact copies (the same `txn_id` repeats). They must be dropped *before* any split, or the same event lands in train and test( cause leakage)

**Validity violations.** `amount` is a string (`"1,500/-"`, `"(227/-)"`, `"4,742/- Dr"`) three different debit encodings; `txn_time` mixes ISO, `dd/mm/yyyy`, and `Mon dd, yyyy` formats. Both need parsing before use.

**Missingness (evidence  mechanism).** Recall Definitions 
 - MCAR = missingness independent of everything
 - MAR = fully explained by *observed* columns; 
 - MNAR = depends on the *unobserved value itself*.


| Column | Missing | Evidence | Class |
|---|---|---|---|
| `agent_id` | ~14% | Missing fraction is ~45% in **Mwanza / Jinja / Arusha** and ~0% everywhere else — missingness is **fully explained by the observed `region`** column (a legacy logger in those regions). | **MAR** |
| `device_id` | ~6% | Missing fraction is flat across region/type and the missing-indicator does **not** move the fraud rate; there is no mechanism tying it to any value. Uniform, uninformative dropout. | **MCAR** |
| `gps_lat` / `gps_lon` | ~36% | Flat across observed columns too (so **not** MAR-explainable), but network/GPS coverage is worse in remote locations, i.e. the probability a coordinate is missing **depends on the coordinate itself**. | **MNAR** |


## 2. Clean and aggregate

- Deduplicate, 
- parse amounts and dates, 
- resolve entities to a canonical `customer_id`
- build customer-level RFM / ratio / cyclical features aligned to `SCORING_TS` (Eq. 2.1, Table 2.2, Listing 2.9).

In [23]:
#- 2a. Exact deduplication 
df = raw.drop_duplicates().reset_index(drop=True)
print("after dedup:", df.shape)

after dedup: (180000, 18)


In [24]:
raw[['amount','txn_id']].sample(30)

,amount,txn_id
7289,"1,232/-",TXN0041236
150407,"-1,569/-",TXN0105162
110175,"-4,488/-",TXN0114781
36111,"2,882/- Dr",TXN0029588
166989,"1,942/- Dr",TXN0034907
21209,(154/-),TXN0053853
80418,"1,924/-",TXN0025327
152217,"(1,296/-)",TXN0070691
85963,(208/-),TXN0013626
50013,"(1,825/-)",TXN0074734


In [25]:

# -2b. Parse the string amount into a signed numeric amount ---

# debit if it starts with '-', is wrapped in (), or is tagged ' Dr'; credit otherwise.

def parse_amount(s):
    
    s = str(s).strip()
    
    neg = s.startswith("(") or s.startswith("-") or "Dr" in s
    
    digits = re.sub(r"[^0-9]", "", s)          # strip commas, '/-', parens, 'Dr', sign
    
    if digits == "":
        
        return np.nan
    
    v = float(digits)
    
    return -v if neg else v



In [26]:
df["amount_signed"]    = df["amount"].map(parse_amount)  

df[['amount','amount_signed']].sample(30)

,amount,amount_signed
156968,"(1,763/-)",-1763.0
156083,"1,751/-",1751.0
37970,"(2,236/-)",-2236.0
117093,72/- Dr,-72.0
63794,"1,303/- Dr",-1303.0
102545,735/- Dr,-735.0
11454,"(2,058/-)",-2058.0
177679,"5,557/-",5557.0
54309,"1,312/-",1312.0
154557,72/-,72.0


In [27]:
df["amount_signed"]    = df["amount"].map(parse_amount)  
df["amount_abs"]       = df["amount_signed"].abs()



print("amount parse failures:", int(df["amount_signed"].isna().sum()))

assert df["amount_signed"].notna().all(), \
    "Amount parsing produced missing values."

assert df["amount_abs"].notna().all(), \
    "Absolute amount contains missing values."

df[["amount", "amount_signed"]].head(4)

amount parse failures: 0


,amount,amount_signed
0,"1,329/- Dr",-1329.0
1,289/- Dr,-289.0
2,"1,571/-",1571.0
3,599/- Dr,-599.0


In [28]:
df[['txn_id','txn_time']]

,txn_id,txn_time
0,TXN0102856,2026-06-30 14:24:19
1,TXN0155490,"Jan 29, 2026 01:17 PM"
2,TXN0019434,"Sep 16, 2025 12:55 AM"
3,TXN0049426,07/04/2026 11:21
4,TXN0013816,20/12/2025 11:05
...,...,...
179995,TXN0066455,27/11/2025 18:15
179996,TXN0053459,"Jul 18, 2026 07:03 AM"
179997,TXN0010742,05/02/2026 05:33
179998,TXN0049689,24/03/2026 01:18


In [29]:
s = df["txn_time"].astype(str)
s

0           2026-06-30 14:24:19
1         Jan 29, 2026 01:17 PM
2         Sep 16, 2025 12:55 AM
3              07/04/2026 11:21
4              20/12/2025 11:05
                  ...          
179995         27/11/2025 18:15
179996    Jul 18, 2026 07:03 AM
179997         05/02/2026 05:33
179998         24/03/2026 01:18
179999         24/06/2025 21:13
Name: txn_time, Length: 180000, dtype: object

In [30]:
ts = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")
ts

0        2026-06-30 14:24:19
1                        NaT
2                        NaT
3                        NaT
4                        NaT
                 ...        
179995                   NaT
179996                   NaT
179997                   NaT
179998                   NaT
179999                   NaT
Name: txn_time, Length: 180000, dtype: datetime64[ns]

In [31]:
ts.isna()

0         False
1          True
2          True
3          True
4          True
          ...  
179995     True
179996     True
179997     True
179998     True
179999     True
Name: txn_time, Length: 180000, dtype: bool

In [32]:
# Format 2: 15/01/2025 14:30
mask = ts.isna()
ts.loc[mask] = pd.to_datetime(
                s[mask],
                format="%d/%m/%Y %H:%M",
                errors="coerce"
            )

In [33]:
# Convert the transaction time column to text
s = df["txn_time"].astype(str)

# Try parsing the dates using different formats

# Format 1: 2025-01-15 14:30:00

ts = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Format 2: 15/01/2025 14:30
mask = ts.isna()
ts.loc[mask] = pd.to_datetime(
                s[mask],
                format="%d/%m/%Y %H:%M",
                errors="coerce"
            )

# Format 3: Jan 15, 2025 02:30 PM
mask = ts.isna()
ts.loc[mask] = pd.to_datetime(
                    s[mask],
                    format="%b %d, %Y %I:%M %p",
                    errors="coerce"
                )

# Save the parsed dates
df["ts"] = ts

# Check how many dates could not be parsed
print("Number of invalid dates:", df["ts"].isna().sum())

# Check that all transactions happened before the scoring date
print("All transactions before scoring date:",
      (df["ts"] < SCORING_TS).all())

assert df["ts"].notna().all(), \
    "Timestamp parsing produced missing values."

assert (df["ts"] < SCORING_TS).all(), \
    "Transactions after the scoring timestamp remain in the dataset."

Number of invalid dates: 0
All transactions before scoring date: True


In [34]:
df[['txn_time','ts']].dtypes

txn_time            object
ts          datetime64[ns]
dtype: object

In [35]:
df["reg_id"].nunique()

11722

In [36]:
df['reg_id'].unique()

array(['NID101089', 'NID103658', 'nid101584', ..., 'NID101841',
       '  NID100165 ', '  NID102518 '], shape=(11722,), dtype=object)

In [37]:
# - 2d. Entity resolution canonical customer_id 


# The msisdn (SIM) is NOT the identity: multi-SIM customers own several. The KYC
# registration id (reg_id) is the linking key; every SIM a person registers shares it.
# It is lightly messy (case / whitespace), so canonicalise before grouping.


# Create a clean customer ID
df["customer_id"] = (
                df["reg_id"]
                .astype(str)      # Convert to text
                .str.strip()      # Remove spaces
                .str.upper()      # Convert to uppercase
            )

# Count unique values
print("Unique SIM numbers (msisdn):", df["msisdn"].nunique())
print("Unique raw registration IDs:", df["reg_id"].nunique())
print("Unique cleaned customer IDs:", df["customer_id"].nunique())

#   keying on msisdn (or raw reg_id) would badly over-count customers and scatter a
#   person's SIMs across CV folds (leakage cause 5). We group on customer_id instead.

Unique SIM numbers (msisdn): 4825
Unique raw registration IDs: 11722
Unique cleaned customer IDs: 3991


### Point-in-time scoring assumption

A fixed scoring timestamp of **1 August 2026** is used for the feature-construction exercise. All customer-level engineered features are calculated only from transactions satisfying `ts < SCORING_TS`. The resulting customer-level features are then joined back to the transaction-level modelling table used by the Lab 2 fraud target.

This preserves the notebook's original transaction-level target while ensuring that the engineered customer-history features do not use information after the defined scoring timestamp.

In [38]:
SCORING_TS = pd.Timestamp("2026-08-01 00:00:00")

In [39]:
# Keep only transactions before the scoring date
df = df[df["ts"] < SCORING_TS].copy()

# ----------------------------
# Create time-based features
# ----------------------------

# Extract the hour of the day (0–23)
df["hour"] = df["ts"].dt.hour

# Extract the day of the week (Monday=0, Sunday=6)
df["dow"] = df["ts"].dt.dayofweek

# Extract the day of the month (1–31)
day = df["ts"].dt.day

# ----------------------------
# Create cyclical features
# ----------------------------

# Hour of the day
df["hr_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hr_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Day of the week
df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)

# ----------------------------
# Other useful features
# ----------------------------

# Distance from the nearest payday (1st or 28th of the month)
df["payday_dist"] = np.minimum((day - 1).abs(), (day - 28).abs())

# Reduce the effect of very large transaction amounts
df["log_amt"] = np.log1p(df["amount_abs"])

# Flag whether the GPS location is missing
df["gps_missing"] = df["gps_lat"].isna().astype(int)

In [40]:
# --- 2f. Customer-level RFM / ratio aggregates as-of SCORING_TS (Table 2.2) ---
# All aggregates use only rows with ts < SCORING_TS (enforced above), so no feature


is_debit  = df["amount_signed"] < 0
by        = df.groupby("customer_id")


agg     = pd.DataFrame({
                        "recency_days":   (SCORING_TS - by["ts"].max()).dt.total_seconds() / 86400,    # R
                        "frequency":      by.size(),                                                   # F
                        "monetary_out":   df.assign(o=np.where(is_debit,  df["amount_abs"], 0)).groupby("customer_id")["o"].sum(),  # M
                        "monetary_in":    df.assign(i=np.where(~is_debit, df["amount_abs"], 0)).groupby("customer_id")["i"].sum(),
                        "n_counterparty": by["counterparty"].nunique(),                                                          # stability: distinct counterparties
                        "cashout_ratio":  df.assign(c=(df["txn_type"]=="cashout").astype(int)).groupby("customer_id")["c"].mean(),
                    })


agg["out_in_ratio"] = agg["monetary_out"] / (agg["monetary_in"] + 1.0)  # scale-free ratio
df = df.merge(agg, on="customer_id", how="left")

print("modelling table:", df.shape, "| customers:", df["customer_id"].nunique())

df[["customer_id","recency_days","frequency","monetary_out","monetary_in",
    "n_counterparty","cashout_ratio","out_in_ratio"]].head(3)

modelling table: (180000, 38) | customers: 3991


,customer_id,recency_days,frequency,monetary_out,monetary_in,n_counterparty,cashout_ratio,out_in_ratio
0,NID101089,12.508333,125,158085.0,73259.0,123,0.136000,2.157862
1,NID103658,12.170995,59,60882.0,44395.0,58,0.118644,1.371340
2,NID101584,1.488889,31,33101.0,26404.0,31,0.129032,1.253588


### 2g. Feature Dictionary and Data Lineage

The following customer-level features extend the baseline features created above. They use only information available before the scoring timestamp, following the strict as-of rule:

**`ts < SCORING_TS`**

The features span five families from Table 2.2: **RFM, Capacity, Velocity/Trend, Stability, and Behavioural**, using 7-, 30- and 90-day look-back windows where appropriate. Upstream fields and their write times are recorded to support lineage, reproducibility and leakage prevention.

| Family | Feature | Definition | Window | Upstream fields | Write time | Fairness note |
|---|---|---|---|---|---|---|
| RFM | `recency_days` | Days since the customer's most recent transaction. | Up to scoring time | `customer_id`, `ts` | Entity/registration time; transaction time | Activity may reflect access to financial services. |
| RFM | `txn_count_7d` | Number of customer transactions. | 7d | `customer_id`, `ts` | Entity time; transaction time | Activity may vary with financial access. |
| RFM | `txn_count_30d` | Number of customer transactions. | 30d | `customer_id`, `ts`, `txn_id` | Entity time; transaction time | May differ by income, location or access. |
| RFM | `monetary_in_30d` | Total incoming transaction value. | 30d | `customer_id`, `ts`, `amount_signed`, `amount_abs` | Entity time; transaction time | May proxy for income/economic status. |
| RFM | `monetary_out_30d` | Total outgoing transaction value. | 30d | `customer_id`, `ts`, `amount_signed`, `amount_abs` | Entity time; transaction time | May reflect socioeconomic differences. |
| RFM | `monetary_in_90d` | Total incoming transaction value. | 90d | `customer_id`, `ts`, `amount_signed`, `amount_abs` | Entity time; transaction time | May proxy for income/financial capacity. |
| Capacity | `out_in_ratio_30d` | Outgoing value relative to incoming value. | 30d | `monetary_in_30d`, `monetary_out_30d` | Derived from transaction-time fields | May reflect differing financial circumstances. |
| Capacity | `savings_rate_30d` | Net value retained relative to incoming value: `(inflow - outflow)/(inflow + 1)`. | 30d | `monetary_in_30d`, `monetary_out_30d` | Derived from transaction-time fields | May correlate with socioeconomic status. |
| Capacity | `cashout_share_30d` | Share of transactions classified as cash-outs. | 30d | `customer_id`, `ts`, `txn_type` | Entity time; transaction time | May vary by occupation or financial access. |
| Velocity/Trend | `balance_slope_90d` | Linear trend in `balance_after`; positive = increasing balance, negative = declining. | 90d | `customer_id`, `ts`, `balance_after` | Entity time; transaction time | Balance trends may reflect economic circumstances. |
| Velocity/Trend | `txn_velocity_ratio_30_vs_prior60d` | Recent transaction activity relative to the preceding 60 days. | 30d vs prior 60d | `txn_count_30d`, `txn_count_90d` | Derived from transaction-time fields | Activity changes may reflect financial access. |
| Stability | `weekly_inflow_std_90d` | Standard deviation of weekly incoming transaction values. | 90d | `customer_id`, `ts`, `amount_signed`, `amount_abs` | Entity time; transaction time | Income instability may correlate with socioeconomic factors. |
| Stability | `n_counterparty_90d` | Number of distinct counterparties. | 90d | `customer_id`, `counterparty`, `ts` | Entity time; transaction time | May proxy for social/economic networks or location. |
| Stability | `agent_diversity_90d` | Number of distinct agents used. | 90d | `customer_id`, `agent_id`, `ts` | Entity time; transaction time | Agent availability may vary geographically. |
| Behavioural | `night_txn_share_30d` | Share of transactions occurring during the defined night period. | 30d | `customer_id`, `ts`, transaction hour | Entity time; transaction time | May proxy for age, occupation, gender or neighbourhood. |

#### Feature-engineering controls

- **Point-in-time:** Only records with `ts < scoring_timestamp` are used.
- **Customer-level:** Features are aggregated to the customer prediction unit.
- **Look-back windows:** 7-, 30- and 90-day windows capture short-, medium- and longer-term behaviour.
- **Lineage:** Each feature is linked to its upstream fields and their availability/write time.
- **Missingness:** Undefined features caused by insufficient historical activity are distinguished from source-data quality problems; final imputation is performed within the modelling pipeline.
- **Fairness:** Financial-capacity and behavioural features are reviewed for potential proxy discrimination before deployment.

These features cover the RFM, Capacity, Velocity/Trend, Stability and Behavioural families specified in Table 2.2.

In [41]:
# 2g. Multi-window customer-level features
# All customer-history features use only transactions strictly before SCORING_TS.
# The resulting customer-level features are joined to the transaction-level modelling table.

WINDOWS = [7, 30, 90]
window_features = {}

for days in WINDOWS:

    cutoff = SCORING_TS - pd.Timedelta(days=days)

    w = df[
        (df["ts"] >= cutoff) &
        (df["ts"] < SCORING_TS)
    ].copy()

    by_customer = w.groupby("customer_id")

    # --------------------------------------------------
    # RFM features
    # --------------------------------------------------

    window_features[f"txn_count_{days}d"] = (
        by_customer.size()
    )

    window_features[f"monetary_in_{days}d"] = (
        w.loc[w["amount_signed"] > 0]
        .groupby("customer_id")["amount_abs"]
        .sum()
    )

    window_features[f"monetary_out_{days}d"] = (
        w.loc[w["amount_signed"] < 0]
        .groupby("customer_id")["amount_abs"]
        .sum()
    )

    # --------------------------------------------------
    # 30-day capacity / behavioural features
    # --------------------------------------------------

    if days == 30:

        inflow = window_features["monetary_in_30d"]
        outflow = window_features["monetary_out_30d"]

        # Outflow relative to inflow
        window_features["out_in_ratio_30d"] = (
            outflow / (inflow + 1.0)
        )

        # Net amount retained relative to inflows
        window_features["savings_rate_30d"] = (
            (inflow - outflow) / (inflow + 1.0)
        )

        total_txns = by_customer.size()

        # Cash-out transaction share
        cashout_txns = (
            w.assign(
                cashout=(
                    w["txn_type"]
                    .astype(str)
                    .str.lower()
                    .eq("cashout")
                ).astype(int)
            )
            .groupby("customer_id")["cashout"]
            .sum()
        )

        window_features["cashout_share_30d"] = (
            cashout_txns / (total_txns + 1e-8)
        )

        # Night-time transaction share
        night_txns = (
            w.assign(
                night=(w["hour"] >= 20).astype(int)
            )
            .groupby("customer_id")["night"]
            .sum()
        )

        window_features["night_txn_share_30d"] = (
            night_txns / (total_txns + 1e-8)
        )

    # --------------------------------------------------
    # 90-day stability features
    # --------------------------------------------------

    if days == 90:

        # Counterparty diversity
        window_features["n_counterparty_90d"] = (
            by_customer["counterparty"].nunique()
        )

        # Agent diversity
        window_features["agent_diversity_90d"] = (
            by_customer["agent_id"].nunique()
        )

        # Weekly inflow variability
        weekly = (
            w.loc[w["amount_signed"] > 0]
            .assign(
                week=w.loc[
                    w["amount_signed"] > 0, "ts"
                ].dt.to_period("W")
            )
            .groupby(["customer_id", "week"])["amount_abs"]
            .sum()
        )

        window_features["weekly_inflow_std_90d"] = (
            weekly.groupby("customer_id").std()
        )

# Combine all customer-level features
window_df = pd.DataFrame(window_features)

# Merge onto the existing dataframe
df = df.merge(
    window_df,
    left_on="customer_id",
    right_index=True,
    how="left"
)

print("Engineered window features:")
print(window_df.columns.tolist())

print("\nFeature matrix shape:", window_df.shape)

Engineered window features:
['txn_count_7d', 'monetary_in_7d', 'monetary_out_7d', 'txn_count_30d', 'monetary_in_30d', 'monetary_out_30d', 'out_in_ratio_30d', 'savings_rate_30d', 'cashout_share_30d', 'night_txn_share_30d', 'txn_count_90d', 'monetary_in_90d', 'monetary_out_90d', 'n_counterparty_90d', 'agent_diversity_90d', 'weekly_inflow_std_90d']

Feature matrix shape: (3819, 16)


In [42]:
# 2h. Velocity and trend features

def calculate_balance_slope(group):

    cutoff = SCORING_TS - pd.Timedelta(days=90)

    w = (
        group[
            (group["ts"] >= cutoff) &
            (group["ts"] < SCORING_TS)
        ]
        .sort_values("ts")
        .copy()
    )

    if len(w) < 2:
        return np.nan

    w["days_from_start"] = (
        w["ts"] - cutoff
    ).dt.total_seconds() / 86400

    try:
        slope, _ = np.polyfit(
            w["days_from_start"].values,
            w["balance_after"].values,
            1
        )

        return slope

    except (TypeError, ValueError):
        return np.nan


# Calculate customer-level 90-day balance trend
balance_slopes = (
    df.groupby("customer_id")
      .apply(calculate_balance_slope)
      .rename("balance_slope_90d")
)

df = df.merge(
    balance_slopes,
    left_on="customer_id",
    right_index=True,
    how="left"
)


# --------------------------------------------------
# Transaction velocity:
# recent 30 days relative to preceding 60 days
# --------------------------------------------------

df["txn_velocity_ratio_30_vs_prior60d"] = (
    df["txn_count_30d"] /
    (
        (df["txn_count_90d"] - df["txn_count_30d"])
        + 1.0
    )
)


print("Velocity/trend features added:")
print([
    "balance_slope_90d",
    "txn_velocity_ratio_30_vs_prior60d"
])

Velocity/trend features added:
['balance_slope_90d', 'txn_velocity_ratio_30_vs_prior60d']


In [43]:
# 2i. Feature engineering validation

ASSIGNMENT_FEATURES = [
    "recency_days",
    "txn_count_7d",
    "txn_count_30d",
    "monetary_in_30d",
    "monetary_out_30d",
    "monetary_in_90d",
    "out_in_ratio_30d",
    "savings_rate_30d",
    "cashout_share_30d",
    "balance_slope_90d",
    "txn_velocity_ratio_30_vs_prior60d",
    "weekly_inflow_std_90d",
    "n_counterparty_90d",
    "agent_diversity_90d",
    "night_txn_share_30d"
]

print("Number of assignment features:", len(ASSIGNMENT_FEATURES))

missing_features = [
    col for col in ASSIGNMENT_FEATURES
    if col not in df.columns
]

print("Missing engineered features:", missing_features)

assert not missing_features, \
    f"Missing assignment features: {missing_features}"

assert set(ASSIGNMENT_FEATURES).issubset(df.columns), \
    "Not all required assignment features are present."

print("\nMissingness:")
print(
    df[ASSIGNMENT_FEATURES]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nFeature summary:")
display(df[ASSIGNMENT_FEATURES].describe().T)

Number of assignment features: 15
Missing engineered features: []

Missingness:
txn_count_7d                         0.476922
out_in_ratio_30d                     0.425022
savings_rate_30d                     0.425022
monetary_in_30d                      0.352672
weekly_inflow_std_90d                0.240978
monetary_out_30d                     0.165478
txn_count_30d                        0.093128
night_txn_share_30d                  0.093128
cashout_share_30d                    0.093128
txn_velocity_ratio_30_vs_prior60d    0.093128
monetary_in_90d                      0.085989
balance_slope_90d                    0.030006
n_counterparty_90d                   0.008100
agent_diversity_90d                  0.008100
recency_days                         0.000000
dtype: float64

Feature summary:


,count,mean,std,min,25%,50%,75%,max
recency_days,180000.0,12.247495,18.214844,0.075208,2.993056,6.497558,14.297257,471.828924
txn_count_7d,94154.0,1.574081,0.940758,1.000000,1.000000,1.000000,2.000000,7.000000
txn_count_30d,163237.0,4.142780,2.773251,1.000000,2.000000,4.000000,6.000000,15.000000
monetary_in_30d,116519.0,3953.874879,3561.991044,29.000000,1431.000000,2879.000000,5354.000000,21546.000000
monetary_out_30d,150214.0,4873.846379,4173.392870,8.000000,1772.000000,3725.000000,6861.000000,27036.000000
monetary_in_90d,164522.0,8494.307077,6685.259579,29.000000,3380.000000,6892.000000,11951.000000,53073.000000
out_in_ratio_30d,103496.0,3.290963,10.828393,0.002347,0.565051,1.294889,2.926051,291.659574
savings_rate_30d,103496.0,-2.291613,10.829304,-290.680851,-1.926426,-0.295341,0.434630,0.997392
cashout_share_30d,163237.0,0.161287,0.231247,0.000000,0.000000,0.000000,0.250000,1.000000
balance_slope_90d,174599.0,-41.601245,2607.851149,-212144.501980,-47.412993,-2.755739,46.538438,17903.677559


## 3. Hunt the leak

A two-minute correlation screen  rank features by |ρ(f, y)| and interrogate
the top of the list. 

In [44]:
# --- 3a. Correlation screen over numeric candidates 
num_candidates = ["manual_review_score", "amount_abs", "log_amt",
                  "hr_sin", "hr_cos", "dow_sin", "dow_cos", "payday_dist", "gps_missing",
                  "recency_days", "frequency", "monetary_out", "monetary_in",
                  "n_counterparty", "cashout_ratio", "out_in_ratio"]


y = df["is_fraud"].values


corr = pd.Series({c: abs(np.corrcoef(df[c].fillna(df[c].median()), y)[0, 1]) for c in num_candidates}).sort_values(ascending=False)


print("|corr| with is_fraud:")
print(corr.round(3).to_string())

# manual_review_score ~0.98 <-- statistical smoke.

|corr| with is_fraud:
manual_review_score    0.981
hr_cos                 0.071
hr_sin                 0.032
log_amt                0.022
amount_abs             0.016
cashout_ratio          0.010
monetary_out           0.004
frequency              0.003
n_counterparty         0.003
monetary_in            0.003
payday_dist            0.002
out_in_ratio           0.002
dow_sin                0.002
recency_days           0.002
gps_missing            0.000
dow_cos                0.000


In [45]:
# --- 3b. The categorical suspect: settlement_status class separation ---

print("fraud rate by settlement_status:")
print(df.groupby("settlement_status")["is_fraud"].mean().round(3).to_string())

print("\ncounts:")
print(df["settlement_status"].value_counts().to_string())
# 'reversed' -> 100% fraud, 'held' -> ~65%, 'settled'/'pending' -> ~0%: near-perfect split.

fraud rate by settlement_status:
settlement_status
held        0.651
pending     0.000
reversed    1.000
settled     0.004

counts:
settlement_status
settled     154760
reversed      9903
pending       8241
held          7096


In [46]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold

NUMERICAL_COLS = [
    "recency_days",
    "txn_count_7d",
    "txn_count_30d",
    "monetary_in_30d",
    "monetary_out_30d",
    "monetary_in_90d",
    "out_in_ratio_30d",
    "savings_rate_30d",
    "cashout_share_30d",
    "balance_slope_90d",
    "txn_velocity_ratio_30_vs_prior60d",
    "weekly_inflow_std_90d",
    "n_counterparty_90d",
    "agent_diversity_90d",
    "night_txn_share_30d"
]

CATEGORY_COLS = ["txn_type", "region", "segment"]

LEAK_NUM   = ["manual_review_score"]
LEAK_CAT   = ["settlement_status"]

def build_model(num_cols, cat_cols):
    num = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("power",  PowerTransformer(method="yeo-johnson")),  # transform to Gaussian-like for LR
        ("scale",  RobustScaler()),# scaling
    ])
    cat = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="UNK")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)),
    ])
    prep = ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])
    
    results= Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000))]) # Pipeline with preprocessing and classifier
    
    return results

y       = df["is_fraud"].values
groups  = df["customer_id"].values           # split by resolved customer, never by SIM
cv      = GroupKFold(n_splits=5)

In [47]:
# --- 3c. Leakage ablation: quantify each planted leakage source ---

# Baseline: neither leakage source included
baseline_model = build_model(NUMERICAL_COLS, CATEGORY_COLS)

# Leakage source 1: manual_review_score only
manual_model = build_model(
    NUMERICAL_COLS + ["manual_review_score"],
    CATEGORY_COLS
)

# Leakage source 2: settlement_status only
settlement_model = build_model(
    NUMERICAL_COLS,
    CATEGORY_COLS + ["settlement_status"]
)

# Both leakage sources
both_model = build_model(
    NUMERICAL_COLS + ["manual_review_score"],
    CATEGORY_COLS + ["settlement_status"]
)

models = {
    "Neither leak": baseline_model,
    "manual_review_score only": manual_model,
    "settlement_status only": settlement_model,
    "Both leaks": both_model
}

ablation_results = {}

for name, model in models.items():

    scores = cross_val_score(
        model,
        df,
        y,
        cv=cv,
        groups=groups,
        scoring="roc_auc"
    )

    ablation_results[name] = {
        "mean_auc": scores.mean(),
        "std_auc": scores.std()
    }

ablation_results = pd.DataFrame(ablation_results).T

display(ablation_results)
 
baseline_auc = ablation_results.loc["Neither leak", "mean_auc"]

print("\nAUC inflation relative to the honest model:")

for name in ablation_results.index:
    if name != "Neither leak":
        inflation = (
            ablation_results.loc[name, "mean_auc"]
            - baseline_auc
        )
        print(f"{name}: {inflation:+.4f}")

,mean_auc,std_auc
Neither leak,0.590931,4.582286e-03
manual_review_score only,1.000000,0.000000e+00
settlement_status only,0.982819,1.965532e-03
Both leaks,1.000000,1.196552e-08



AUC inflation relative to the honest model:
manual_review_score only: +0.4091
settlement_status only: +0.3919
Both leaks: +0.4091


### 3d. Impact of the leakage

The honest model achieves a cross-validated ROC-AUC of **0.5909 ± 0.0046**. Including `manual_review_score` and `settlement_status` raises the ROC-AUC to **1.0000 ± 0.0000**, an inflation of **+0.4091 AUC**.

The ablation confirms that each planted leak is individually responsible for a large performance increase: `manual_review_score` increases AUC by **+0.4091**, while `settlement_status` increases it by **+0.3919**.

The apparently perfect score is therefore not evidence of a highly predictive model; it is evidence that post-outcome information has entered the feature set.

### 3e. Leakage mechanism and governance controls

Both planted leakage sources are primarily **Cause 6: leakage from the data-collection process**. They are operational fields whose values become available after the transaction/outcome and therefore were not available at the scoring timestamp.

- **`manual_review_score`**: the field reflects a downstream manual review process. Using it as a predictor allows information generated after the relevant event/outcome to enter model training.
- **`settlement_status`**: the status reflects a downstream settlement process. Values such as `reversed` and `held` are strongly associated with the observed fraud outcome, indicating that the field contains information generated after or as a consequence of the event being predicted.

The practical prevention is to enforce point-in-time feature eligibility using field write timestamps and maintain feature-level lineage. Any field written after the scoring timestamp, or triggered by the outcome or its downstream operational handling, should be excluded from model inputs. A feature review and approval process should also require documented data lineage before a feature enters model development.|


## 4. Reproducible Machine Learning Pipeline

The final model is implemented as a single Scikit-learn `Pipeline` containing all model-fitting preprocessing steps and the classifier. Numerical variables are imputed, transformed and scaled, while categorical variables are imputed and one-hot encoded.

The model is evaluated using 5-fold `GroupKFold`, with the resolved `customer_id` used as the grouping variable to prevent observations from the same customer appearing in different folds.

The leakage variables `manual_review_score` and `settlement_status` are excluded from the final model because they contain post-outcome information that would not be available at scoring time.

`GroupKFold` is used to satisfy the group-aware evaluation requirement and prevent customer-level leakage across folds. A production deployment would additionally use time-ordered evaluation so that future observations cannot inform earlier scoring periods.

In [48]:
# --- 4a. Build the final leakage-safe pipeline ---

final_model = build_model(
    NUMERICAL_COLS,
    CATEGORY_COLS
)

print(final_model)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('power',
                                                                   PowerTransformer()),
                                                                  ('scale',
                                                                   RobustScaler())]),
                                                  ['recency_days',
                                                   'txn_count_7d',
                                                   'txn_count_30d',
                                                   'monetary_in_30d',
                                        

In [49]:
# --- 4b. Group-aware cross-validation ---

auc_scores = cross_val_score(
    final_model,
    df,
    y,
    cv=cv,
    groups=groups,
    scoring="roc_auc"
)

print(
    f"Final cross-validated ROC-AUC: "
    f"{auc_scores.mean():.4f} (+/- {auc_scores.std():.4f})"
)

Final cross-validated ROC-AUC: 0.5909 (+/- 0.0046)


In [50]:
print("\nFold-level ROC-AUC:")
for i, score in enumerate(auc_scores, start=1):
    print(f"Fold {i}: {score:.4f}")


Fold-level ROC-AUC:
Fold 1: 0.5973
Fold 2: 0.5934
Fold 3: 0.5894
Fold 4: 0.5912
Fold 5: 0.5834


## 5. Pipeline Serialization and Reload Verification

The final leakage-safe Pipeline is fitted and serialized as a single `joblib` artifact. Because the preprocessing steps are contained within the Pipeline, the saved object preserves the preprocessing and model together. The artifact is then reloaded and tested to confirm that it can generate the same predictions as the original fitted Pipeline without additional preprocessing or modification.

In [51]:
# --- 5a. Fit the final pipeline on the development data ---

final_model.fit(df, y)

print("Final pipeline fitted successfully.")

Final pipeline fitted successfully.


In [52]:
# --- 5b. Serialize the final pipeline ---

import joblib

MODEL_PATH = "credit_risk_pipeline.joblib"

joblib.dump(
    final_model,
    MODEL_PATH
)

print(f"Pipeline saved successfully: {MODEL_PATH}")

Pipeline saved successfully: credit_risk_pipeline.joblib


In [53]:
# --- 5c. Reload the saved pipeline ---

loaded_model = joblib.load(MODEL_PATH)

print("Pipeline reloaded successfully.")

Pipeline reloaded successfully.


In [54]:
# --- 5d. Verify the reloaded pipeline can generate predictions ---

sample = df.head(10)

original_predictions = final_model.predict_proba(sample)[:, 1]
reloaded_predictions = loaded_model.predict_proba(sample)[:, 1]

print("Original predictions:")
print(original_predictions)

print("\nReloaded predictions:")
print(reloaded_predictions)

Original predictions:
[0.09375253 0.10688027 0.06114193 0.09707548 0.12830878 0.09401853
 0.06614267 0.06734528 0.06760738 0.06647176]

Reloaded predictions:
[0.09375253 0.10688027 0.06114193 0.09707548 0.12830878 0.09401853
 0.06614267 0.06734528 0.06760738 0.06647176]


In [55]:
# Confirm that serialization did not change the predictions

assert np.allclose(
    original_predictions,
    reloaded_predictions
)

print("\nPipeline reload verification: PASSED")


Pipeline reload verification: PASSED


### Reproducibility and Deployment

The final leakage-safe pipeline achieves a cross-validated ROC-AUC of **0.5909 ± 0.0046** using 5-fold `GroupKFold`. In contrast, including the two post-outcome leakage variables increases ROC-AUC to **1.0000**, an inflation of **+0.4091 AUC**. This demonstrates that the apparently perfect offline performance is caused by information that would not be available when the model is used for scoring.

The final Pipeline contains all model-fitting preprocessing steps, including imputation, transformation, scaling and categorical encoding, so these operations are refitted within each cross-validation fold. Grouping by the resolved `customer_id` also prevents the same customer's records from being split across folds.

Finally, the fitted Pipeline was serialized with `joblib`, reloaded successfully, and produced identical predictions to the original fitted Pipeline. This confirms that the submitted model artifact is reproducible and executable without manually repeating the preprocessing steps.